# Larger open-weight LLM (14B), run locally

Qwen2.5-14B-Instruct in 4-bit on the free T4, with the same prompts and examples as notebook 06, to test whether model size closes the gap. Runs the few-shot-with-position setting by default. T4 GPU; first run downloads about 30 GB.

In [ ]:
# ===== STEP 6b: a larger open-weight LLM (14B) running locally - does scale close the gap? =====
# Runtime > Change runtime type > T4 GPU. First run downloads about 30 GB (15 to 25 minutes), then about 30 to 40 minutes
# for the one condition run by default (few-shot with position, the 7B model's best setting).
!pip -q install -U bitsandbytes accelerate

import os, json, time, re
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import f1_score, accuracy_score, classification_report
from google.colab import drive
drive.mount("/content/drive")

DATA = "/content/drive/MyDrive/sci-classify/data"; RES = "/content/drive/MyDrive/sci-classify/results"
LABELS = ["BACKGROUND", "OBJECTIVE", "METHODS", "RESULTS", "CONCLUSIONS"]
test_llm = pd.read_parquet(f"{DATA}/test_llm_1000.parquet")
train = pd.read_parquet(f"{DATA}/train.parquet")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch to a T4 runtime")

# Twice the size of the 7B model in Step 6, same prompts and examples, so only the model size changes.
CANDIDATES = [("Qwen/Qwen2.5-14B-Instruct", True)]      # about 9-10 GB in 4-bit: fits the 16 GB T4
for mid, four_bit in CANDIDATES:
    try:
        tok = AutoTokenizer.from_pretrained(mid)
        kw = {"quantization_config": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                                        bnb_4bit_quant_type="nf4")} if four_bit else {"dtype": torch.float16}
        model = AutoModelForCausalLM.from_pretrained(mid, device_map="auto", low_cpu_mem_usage=True, **kw); model.eval()
        MODEL = mid + (" (4-bit)" if four_bit else " (fp16)"); break
    except Exception as e:
        print("Could not load", mid, "->", str(e)[:150]); torch.cuda.empty_cache()
assert "MODEL" in globals(), "The 14B model did not load - paste the error above to Claude"
print("Using:", MODEL, f"| GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

BATCH = 20
INSTRUCTIONS = (
    "You classify sentences from abstracts of randomised controlled trials by their rhetorical role.\n"
    "Labels:\n"
    "BACKGROUND: context, prior knowledge or the problem being addressed.\n"
    "OBJECTIVE: the aim, purpose or hypothesis of this study.\n"
    "METHODS: design, participants, interventions, measurements or analysis.\n"
    "RESULTS: findings, numbers and statistical outcomes.\n"
    "CONCLUSIONS: interpretation, implications or recommendations.\n"
    "Return only a JSON list of labels, one per sentence, in the same order.")

def few_shot_block():                                  # identical examples to the Gemini step
    ex = train.groupby("label").sample(n=2, random_state=7)
    return "Examples:\n" + "\n".join(f"[{r.line_number + 1} of {r.total_lines}] {r.text} -> {r.label}"
                                     for r in ex.itertuples()) + "\n\n"

def build_prompt(rows, use_position, few_shot):
    items = [f"{i + 1}. " + (f"[{r.line_number + 1} of {r.total_lines}] " if use_position else "") + r.text
             for i, r in enumerate(rows.itertuples())]
    pos_note = "Each sentence starts with its position in its abstract, e.g. [3 of 9].\n" if use_position else ""
    return INSTRUCTIONS + "\n" + pos_note + "\n" + (few_shot_block() if few_shot else "") + \
           "Sentences:\n" + "\n".join(items)

@torch.no_grad()
def generate(prompt, n):
    msgs = [{"role": "user", "content": prompt}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    if not isinstance(enc, dict) and not hasattr(enc, "keys"):      # older transformers return a plain tensor
        enc = {"input_ids": enc}
    enc = {k: v.to(model.device) for k, v in enc.items()}
    out = model.generate(**enc, max_new_tokens=12 * n + 20, do_sample=False, pad_token_id=tok.eos_token_id)
    text = tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(r"\[.*?\]", text, re.S)                     # first JSON list in the reply
    try:
        labels = [str(x).strip().upper() for x in json.loads(m.group(0))] if m else []
    except Exception:
        labels = []
    if len(labels) != n:                                       # a malformed answer counts as wrong: that is the model's failure
        labels = (labels + ["INVALID"] * n)[:n]
    return [l if l in LABELS else "INVALID" for l in labels]

def run(name, use_position, few_shot):
    path = f"{RES}/step6b_{name}.json"
    done = json.load(open(path)) if os.path.exists(path) else {}
    t0 = time.time()
    for start in range(0, len(test_llm), BATCH):
        if str(start) in done:
            continue
        rows = test_llm.iloc[start:start + BATCH]
        done[str(start)] = generate(build_prompt(rows, use_position, few_shot), len(rows))
        json.dump(done, open(path, "w"))
        print(f"{name}: {min(start + BATCH, len(test_llm))}/{len(test_llm)}", end="\r")
    preds = [p for k in sorted(done, key=int) for p in done[k]]
    secs = time.time() - t0
    f1 = f1_score(test_llm.label, preds, average="macro", labels=LABELS)
    print(f"\n{name:34s} macro-F1 {f1:.3f}  accuracy {accuracy_score(test_llm.label, preds):.3f}  "
          f"invalid {preds.count('INVALID')}  ({secs / len(test_llm) * 1000:.0f} ms/sentence this session)")
    timing[name] = round(secs / len(test_llm) * 1000) if secs > 60 else timing.get(name)
    return preds

timing = {}
results = {}
CONDITIONS = [("open_llm_few_shot_position", True, True)]
# To also run the zero-shot settings (adds about an hour), use:
# CONDITIONS = [("open_llm_zero_shot", False, False), ("open_llm_zero_shot_position", True, False),
#               ("open_llm_few_shot_position", True, True)]
for name, pos, fs in CONDITIONS:
    results[name] = run(name, pos, fs)

best = max(results, key=lambda k: f1_score(test_llm.label, results[k], average="macro", labels=LABELS))
print(f"\nPer-class F1 for the 14B model ({best}):")
print(classification_report(test_llm.label, results[best], labels=LABELS, digits=3, zero_division=0))
json.dump({"model": MODEL, "preds": results, "ms": timing}, open(f"{RES}/step6b_open_llm_14b.json", "w"))
print("Saved to", RES)
